# Operationally Grounded Early Warning for Noisy Quantum Communication Links
### Reproduction notebook — ARRAY-D-26-02374 (revised)

Runs every experiment reported in the manuscript in one pass and writes all figures,
tables and a machine-readable manifest to Google Drive.

**Sections**
1. Configuration and Drive mount
2. Exact link model (Kraus) + verification against closed forms
3. Operational quantities and the four independent failure events
4. Exact critical points (Table 1)
5. Capacity proxy vs. exact coherent information (Fig. 2)
6. Trajectory generator
7. Matched-false-alarm evaluation (Table 2, Fig. 3)
8. Ablation (Table 3, Fig. 4)
9. Parameter sensitivity (Table 4)
10. Critical-slowing-down precursors (Table 5, Fig. 6)
11. Measurement-aware implementation (Table 6, Fig. 7)
12. Extended link: loss, dark counts, swapping (Fig. 8)
13. Manifest and completeness check

Every number in the manuscript comes from one execution of this notebook.


## 1. Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/Outputs/QuantumEarlyWarning/"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)

In [ ]:
!pip -q install qiskit 2>/dev/null | tail -1

import numpy as np, pandas as pd, json, platform, sys
from functools import reduce
from scipy.optimize import brentq
from scipy import stats
import matplotlib; import matplotlib.pyplot as plt

# ---- fixed, disclosed parameters (Section 3.4) ----
PAR = dict(alpha=0.35, beta=0.35, gamma=0.20, delta=0.10, kappa=3.0, eta=0.5, Dmax=1.5)
SEED_MAIN, SEED_AUX = 770126, 990126
T = 120                       # monitoring steps per trajectory
N_DEG, N_HEAL = 200, 200      # main evaluation
FARS = [0.02, 0.05, 0.10]
BOOT = 3000
print(PAR); print("seeds", SEED_MAIN, SEED_AUX)

### 1b. Re-entry guard
If the Colab kernel restarts mid-run, execute this cell and then continue from any section.

In [ ]:
def ensure_state():
    g = globals()
    need = ['np','pd','json','plt','PAR','link_state','skr','EVENTS']
    missing = [n for n in need if n not in g]
    if missing:
        print("missing:", missing, "-> re-run sections 1 to 6")
    else:
        print("state OK")
    return not missing
ensure_state()

## 2. Exact link model
An ideal Bell pair is prepared, then **one subsystem is transmitted** through the channel (Reviewer 3, Comment 2). All parameters confined to [0,1] (Reviewer 3, Comment 3).

In [ ]:
"""Exact density-matrix link model: ideal Bell pair, channel on transmitted qubit only."""
import numpy as np
from functools import reduce

I2=np.eye(2,dtype=complex)
X=np.array([[0,1],[1,0]],dtype=complex); Y=np.array([[0,-1j],[1j,0]]); Z=np.diag([1,-1]).astype(complex)

def kron(*a): return reduce(np.kron,a)

PHI=np.zeros(4,dtype=complex); PHI[0]=PHI[3]=1/np.sqrt(2)
RHO_REF=np.outer(PHI,PHI.conj())

# ---- single-qubit channel Kraus sets ----
def K_depol(p):
    # standard depolarizing: rho -> (1-p)rho + p I/2
    return [np.sqrt(1-3*p/4)*I2, np.sqrt(p/4)*X, np.sqrt(p/4)*Y, np.sqrt(p/4)*Z]
def K_phase(p):          # phase damping / dephasing, p = prob of Z error*2 convention
    return [np.sqrt(1-p/2)*I2, np.sqrt(p/2)*Z]
def K_amp(g):            # amplitude damping, gamma
    return [np.array([[1,0],[0,np.sqrt(1-g)]],dtype=complex),
            np.array([[0,np.sqrt(g)],[0,0]],dtype=complex)]
def K_loss_flag(g):      # photon loss -> treated separately (heralded)
    return K_amp(g)

def compose(*sets):
    out=[np.eye(2,dtype=complex)]
    for S in sets:
        out=[k@m for k in S for m in out]
    return out

def apply_on_B(rho, Ks):
    """apply single-qubit Kraus set to qubit B (second tensor factor)."""
    return sum(kron(I2,k)@rho@kron(I2,k).conj().T for k in Ks)

def chan(pd,pp,pa):
    return compose(K_depol(pd),K_phase(pp),K_amp(pa))

def link_state(pd,pp,pa):
    """Ideal Bell pair prepared noiselessly; qubit B transmitted through the channel."""
    return apply_on_B(RHO_REF, chan(pd,pp,pa))

# ---- quantities ----
def _ev(rho):
    w=np.linalg.eigvalsh((rho+rho.conj().T)/2)
    return w[w>1e-12]
def vn(rho):
    w=_ev(rho); return float(-(w*np.log2(w)).sum())
def ptrace_B(rho):  # keep A
    r=rho.reshape(2,2,2,2); return np.trace(r,axis1=1,axis2=3)
def ptrace_A(rho):  # keep B
    r=rho.reshape(2,2,2,2); return np.trace(r,axis1=0,axis2=2)

def S_norm(rho): return vn(rho)/2.0            # log2 dim = 2 for 2 qubits
def fidelity_pure_ref(rho): return float(np.real(PHI.conj()@rho@PHI))

def coherent_info(rho):
    """I_c(A>B) = S(rho_B) - S(rho_AB).  >0 required for quantum capacity / recoverability."""
    return vn(ptrace_A(rho))-vn(rho)

def negativity(rho):
    r=rho.reshape(2,2,2,2).transpose(0,3,2,1).reshape(4,4)   # partial transpose on B
    w=np.linalg.eigvalsh((r+r.conj().T)/2)
    return float(np.abs(w[w<0]).sum())

def concurrence(rho):
    sy=kron(Y,Y)
    R=rho@sy@rho.conj()@sy
    w=np.sort(np.sqrt(np.abs(np.linalg.eigvals(R).real)))[::-1]
    return float(max(0.0, w[0]-w[1]-w[2]-w[3]))

def singlet_fraction(rho):
    """max overlap with a maximally entangled state under local unitaries -> use Bell basis max."""
    b=[np.array([1,0,0,1])/np.sqrt(2), np.array([1,0,0,-1])/np.sqrt(2),
       np.array([0,1,1,0])/np.sqrt(2), np.array([0,1,-1,0])/np.sqrt(2)]
    return max(float(np.real(v.conj()@rho@v)) for v in b)

def tele_fidelity(rho):
    """standard-teleportation fidelity from singlet fraction; classical limit 2/3."""
    return (2*singlet_fraction(rho)+1)/3

def bell_correlators(rho):
    """<XX>,<YY>,<ZZ> for the prepared Phi+ ; QBER in Z and X bases for E91/BB84."""
    exx=float(np.real(np.trace(kron(X,X)@rho)))
    eyy=float(np.real(np.trace(kron(Y,Y)@rho)))
    ezz=float(np.real(np.trace(kron(Z,Z)@rho)))
    return exx,eyy,ezz

def qber(rho):
    exx,eyy,ezz=bell_correlators(rho)
    ez=(1-ezz)/2; ex=(1-exx)/2
    return ez,ex

def h2(x):
    x=min(max(x,1e-12),1-1e-12)
    return -x*np.log2(x)-(1-x)*np.log2(1-x)

def skr(rho):
    """asymptotic BB84/E91 secret-key rate lower bound r = 1 - h(e_z) - h(e_x)."""
    ez,ex=qber(rho)
    return 1-h2(ez)-h2(ex)


In [ ]:
# verification against closed forms
chk = []
for p in [0.1,0.4,0.8]:
    chk.append(dict(p=p, sim=fidelity_pure_ref(link_state(p,0,0)), analytic=1-3*p/4))
vdf = pd.DataFrame(chk); vdf["abs_err"] = (vdf.sim-vdf.analytic).abs()
print(vdf.to_string(index=False))
print("max Werner error:", vdf.abs_err.max())
ed = brentq(lambda p: concurrence(link_state(p,0,0))-1e-12, 1e-9, 0.999, xtol=1e-14)
print("entanglement death: sim %.12f  analytic %.12f  err %.2e" % (ed, 2/3, abs(ed-2/3)))
assert vdf.abs_err.max() < 1e-12 and abs(ed-2/3) < 1e-10
print("VERIFIED")

## 3-6. Descriptors, events, trajectories

In [ ]:
import numpy as np, pandas as pd


T=120
PAR=dict(alpha=0.35,beta=0.35,gamma=0.20,delta=0.10,kappa=3.0,eta=0.5,Dmax=1.5)

def descriptors(rho,pd_,pp_,pa_,P=PAR):
    S=S_norm(rho); L=1-fidelity_pure_ref(rho)
    D=min(pd_+pp_+pa_,P['Dmax'])/P['Dmax']
    U=max(0.0,1-P['eta']*D)
    return S,L,D,U

def qcci(S,L,D,U,P=PAR):
    return float(min(max(P['alpha']*L+P['beta']*S+P['gamma']*D-P['delta']*U,0.0),1.0))

def baseline_classical(pd_,pp_,pa_):
    return float(min(max((pd_+pp_+pa_)/3.0,0.0),1.0))

EVENTS={'skr_collapse':lambda d:d.SKR<=0,'coh_info_nonpos':lambda d:d.Ic<=0,
        'ent_death':lambda d:d.C<=1e-9,'tele_below_classical':lambda d:d.Ftel<=2/3}
def first_true(mask):
    i=np.flatnonzero(mask.values); return int(i[0]) if len(i) else None
def alarm(s,thr):
    i=np.flatnonzero(s.values>=thr); return int(i[0]) if len(i) else None

def make_traj(rng,kind,hetero=True):
    w=rng.dirichlet([0.7,0.7,0.7]) if hetero else np.array([1/3,1/6,0.1])*3
    if kind=='degrading':
        rate=rng.uniform(0.006,0.020); onset=rng.integers(8,35)
        base=np.clip(np.concatenate([np.zeros(onset),rate*np.arange(T-onset)]),0,3)
    else:
        base=np.full(T,rng.uniform(0.02,0.20))
    ou=np.zeros(T); sig=rng.uniform(0.004,0.014)
    for t in range(1,T): ou[t]=ou[t-1]+0.15*(0-ou[t-1])+rng.normal(0,sig)
    L=np.clip(base+ou,0,3)
    return np.clip(L*w[0],0,1),np.clip(L*w[1],0,1),np.clip(L*w[2],0,1)

def run(kind,rng,hetero=True,shots=None):
    pdv,ppv,pav=make_traj(rng,kind,hetero); rec=[]
    for t in range(T):
        r=link_state(pdv[t],ppv[t],pav[t])
        S,L,D,U=descriptors(r,pdv[t],ppv[t],pav[t])
        row=dict(t=t,pd=pdv[t],pp=ppv[t],pa=pav[t],S=S,L=L,D=D,U=U,
            QCCI=qcci(S,L,D,U),ENT=S,FID_L=L,DEC=D,
            BASE=baseline_classical(pdv[t],ppv[t],pav[t]),
            QCCI_state=float(min(max((PAR['alpha']*L+PAR['beta']*S)/(PAR['alpha']+PAR['beta']),0),1)),
            Cproxy=float(np.exp(-PAR['kappa']*S)),
            SKR=skr(r),Ic=coherent_info(r),C=concurrence(r),Ftel=tele_fidelity(r))
        if shots:
            row.update(measure_aware(r,pdv[t],ppv[t],pav[t],shots,rng))
        rec.append(row)
    return pd.DataFrame(rec)

def measure_aware(rho,pd_,pp_,pa_,shots,rng,P=PAR):
    """Finite-shot stabiliser tomography of the Bell pair: <XX>,<YY>,<ZZ> only (3 settings)."""
    est=[]
    for M in [kron(X,X),kron(Y,Y),kron(Z,Z)]:
        p1=(1+float(np.real(np.trace(M@rho))))/2
        k=rng.binomial(shots,min(max(p1,0),1))
        est.append(2*k/shots-1)
    exx,eyy,ezz=est
    # Bell-diagonal reconstruction from the three correlators
    F_hat=(1+exx-eyy+ezz)/4
    lam=np.array([(1+exx-eyy+ezz),(1-exx+eyy+ezz),(1+exx+eyy-ezz),(1-exx-eyy-ezz)])/4
    lam=np.clip(lam,1e-12,1); lam=lam/lam.sum()
    S_hat=float(-(lam*np.log2(lam)).sum())/2
    L_hat=1-float(np.clip(F_hat,0,1))
    D_hat=min(pd_+pp_+pa_,P['Dmax'])/P['Dmax']; U_hat=max(0.0,1-P['eta']*D_hat)
    ez=(1-ezz)/2; ex=(1-exx)/2
    skr_hat=1-h2(min(max(ez,0),1))-h2(min(max(ex,0),1))
    return dict(S_hat=S_hat,L_hat=L_hat,QCCI_hat=qcci(S_hat,L_hat,D_hat,U_hat),
                ENT_hat=S_hat,FID_L_hat=L_hat,SKR_hat=skr_hat)

def lead_at_far(test_d,test_h,ind,ev_fn,target_far,n_grid=400):
    """Choose the threshold that realises exactly target_far on held-out healthy trajectories,
    then report lead time on failing trajectories. Fully matched-FAR comparison."""
    peaks=np.sort(np.array([d[ind].max() for d in test_h]))
    lo=min(d[ind].min() for d in test_d+test_h); hi=max(d[ind].max() for d in test_d+test_h)
    grid=np.linspace(lo,hi,n_grid)
    fars=np.array([np.mean(peaks>=g) for g in grid])
    ok=np.flatnonzero(fars<=target_far+1e-12)
    thr=float(grid[ok[0]]) if len(ok) else float(hi)
    leads=[];miss=0;nd=0
    for d in test_d:
        tf=first_true(ev_fn(d))
        if tf is None: continue
        nd+=1; ta=alarm(d[ind],thr)
        if ta is None or ta>tf: miss+=1
        else: leads.append(tf-ta)
    return thr,float(np.mean(peaks>=thr)),np.array(leads,float),miss,nd


## 4. Exact critical points — Table 1

In [ ]:
def cross(f, lo=1e-9, hi=1-1e-9):
    try: return float(brentq(f, lo, hi, xtol=1e-12))
    except ValueError: return None

axes = {"depolarizing": lambda p: link_state(p,0,0),
        "phase_damping": lambda p: link_state(0,p,0),
        "amplitude_damping": lambda p: link_state(0,0,p)}
evd  = {"skr_zero": lambda r: skr(r), "coh_info_zero": lambda r: coherent_info(r),
        "concurrence_zero": lambda r: concurrence(r)-1e-12,
        "tele_classical": lambda r: tele_fidelity(r)-2/3,
        "fid_half": lambda r: fidelity_pure_ref(r)-0.5}
rows=[]
for ax,st in axes.items():
    for ev,fn in evd.items():
        rows.append(dict(axis=ax, event=ev, critical=cross(lambda p: fn(st(p)))))
T1 = pd.DataFrame(rows).pivot(index="event", columns="axis", values="critical")
T1.to_csv(OUT_DIR+"table1_critical_points.csv"); print(T1.round(4).to_string())

## 5. Capacity proxy vs. exact coherent information — Fig. 2

In [ ]:
G=np.round(np.linspace(0,1,21),4); rows=[]
for pdv in G:
    for ppv in [0.0,0.25,0.5]:
        for pav in [0.0,0.25,0.5]:
            r=link_state(pdv,ppv,pav)
            rows.append(dict(pd=float(pdv),pp=ppv,pa=pav,S=S_norm(r),F=fidelity_pure_ref(r),
                Ic=coherent_info(r),C=concurrence(r),Ftel=tele_fidelity(r),SKR=skr(r),
                Cproxy=float(np.exp(-PAR["kappa"]*S_norm(r)))))
grid=pd.DataFrame(rows); grid.to_csv(OUT_DIR+"grid_mixed_noise.csv",index=False)
proxy=dict(n=len(grid),
    pearson=float(np.corrcoef(grid.Cproxy,grid.Ic)[0,1]),
    spearman=float(grid.Cproxy.corr(grid.Ic,method="spearman")),
    max_gap=float(np.max(np.abs(grid.Cproxy-((grid.Ic+1)/2)))))
print(proxy)

## 7. Matched-false-alarm evaluation — Table 2, Fig. 3
This is the principal experiment. Every indicator is calibrated on **held-out healthy** trajectories to the same realised false-alarm rate before lead time is measured (Reviewer 3, Comment 1).

In [ ]:
rng = np.random.default_rng(SEED_MAIN)
print("simulating trajectories ...")
test_h = [run("healthy",  rng) for _ in range(N_HEAL)]
test_d = [run("degrading",rng) for _ in range(N_DEG)]
IND = ["QCCI","ENT","FID_L","DEC","BASE","QCCI_state"]
print("done:", len(test_d),"degrading /",len(test_h),"healthy")

In [ ]:
results={"params":PAR,"seed":SEED_MAIN,"n_deg":N_DEG,"n_heal":N_HEAL,
          "verification":{"werner_max_err":float(vdf.abs_err.max()),"ent_death_err":float(abs(ed-2/3))},
          "proxy_vs_exact":proxy,"matched_far":{}}
for FAR in FARS:
    results["matched_far"][f"{FAR:.2f}"]={}
    for ev,fn in EVENTS.items():
        row={}
        for ind in IND:
            thr,rf,L,miss,nd = lead_at_far(test_d,test_h,ind,fn,FAR)
            b=[np.mean(rng.choice(L,len(L))) for _ in range(BOOT)] if len(L) else [np.nan]
            row[ind]=dict(threshold=thr,realised_far=rf,n_failing=nd,n_detected=len(L),
                lead_mean=float(L.mean()) if len(L) else None,
                lead_median=float(np.median(L)) if len(L) else None,
                lead_ci=[float(np.percentile(b,2.5)),float(np.percentile(b,97.5))] if len(L) else None,
                missed_rate=miss/max(1,nd))
        results["matched_far"][f"{FAR:.2f}"][ev]=row

T2=pd.DataFrame({i:{ "lead":results["matched_far"]["0.05"]["skr_collapse"][i]["lead_mean"],
                     "missed":results["matched_far"]["0.05"]["skr_collapse"][i]["missed_rate"],
                     "far":results["matched_far"]["0.05"]["skr_collapse"][i]["realised_far"]} for i in IND}).T
T2.to_csv(OUT_DIR+"table2_lead_time.csv"); print(T2.round(4).to_string())

In [ ]:
# paired per-trajectory comparison against QCCI, Holm-corrected
results["paired"]={}
for ev,fn in EVENTS.items():
    thrs={i:lead_at_far(test_d,test_h,i,fn,0.05)[0] for i in IND}
    pr={}
    for ind in IND:
        if ind=="QCCI": continue
        dif=[]
        for d in test_d:
            tf=first_true(fn(d))
            if tf is None: continue
            a=alarm(d["QCCI"],thrs["QCCI"]); b_=alarm(d[ind],thrs[ind])
            ka=(tf-a) if (a is not None and a<=tf) else None
            kb=(tf-b_) if (b_ is not None and b_<=tf) else None
            if ka is None or kb is None: continue
            dif.append(ka-kb)
        dif=np.array(dif,float)
        if len(dif)>2:
            t,p=stats.ttest_1samp(dif,0)
            w=stats.wilcoxon(dif) if np.any(dif!=0) else None
            pr[ind]=dict(n=len(dif),mean_diff=float(dif.mean()),p_ttest=float(p),
                         p_wilcoxon=float(w.pvalue) if w else None,
                         ci=[float(np.percentile([np.mean(rng.choice(dif,len(dif))) for _ in range(BOOT)],q)) for q in (2.5,97.5)])
    ps=sorted([(v["p_ttest"],k) for k,v in pr.items()]); m=len(ps)
    for i,(p,k) in enumerate(ps): pr[k]["p_holm"]=float(min(1,p*(m-i)))
    results["paired"][ev]=pr
for k,v in results["paired"]["skr_collapse"].items():
    print(f"QCCI - {k:11s} = {v['mean_diff']:+.3f}  p_holm={v['p_holm']:.2e}")

## 8. Ablation — Table 3, Fig. 4

In [ ]:
def qcci_variant(d,keep):
    w={"L":PAR["alpha"],"S":PAR["beta"],"D":PAR["gamma"]}
    tot=sum(w[k] for k in keep if k in w)
    v=sum(w[k]*d[k] for k in keep if k in w)/max(tot,1e-9)
    if "U" in keep: v=v-PAR["delta"]*d["U"]
    return np.clip(v,0,1)
ABL={"full_LSDU":["L","S","D","U"],"no_U":["L","S","D"],"no_D":["L","S","U"],
     "no_S":["L","D","U"],"no_L":["S","D","U"],"L_only":["L"],"S_only":["S"],"D_only":["D"]}
results["ablation"]={}
for ev,fn in EVENTS.items():
    row={}
    for name,keep in ABL.items():
        for d in test_d+test_h: d["_V"]=qcci_variant(d,keep)
        thr,rf,L,miss,nd=lead_at_far(test_d,test_h,"_V",fn,0.05)
        row[name]=dict(realised_far=rf,lead_mean=float(L.mean()) if len(L) else None,
                       missed_rate=miss/max(1,nd))
    results["ablation"][ev]=row
T3=pd.DataFrame(results["ablation"]["skr_collapse"]).T
T3.to_csv(OUT_DIR+"table3_ablation.csv"); print(T3.round(4).to_string())

## 9. Parameter sensitivity — Table 4
Full factorial over the four weights, false-alarm rate rematched at each point (Reviewer 4, Comment 11).

In [ ]:
grid_rows=[]
for a in [0.2,0.35,0.5]:
  for b in [0.2,0.35,0.5]:
    for g in [0.0,0.2,0.4]:
      for dl in [0.0,0.1,0.2]:
        for d in test_d+test_h: d["_W"]=np.clip(a*d.L+b*d.S+g*d.D-dl*d.U,0,1)
        thr,rf,L,miss,nd=lead_at_far(test_d,test_h,"_W",EVENTS["skr_collapse"],0.05)
        grid_rows.append(dict(alpha=a,beta=b,gamma=g,delta=dl,realised_far=rf,
            lead_mean=float(L.mean()) if len(L) else None,missed_rate=miss/max(1,nd)))
SW=pd.DataFrame(grid_rows); SW.to_csv(OUT_DIR+"table4_weight_sensitivity.csv",index=False)
results["sensitivity"]={"weights":dict(n=len(SW),lead_min=float(SW.lead_mean.min()),
    lead_max=float(SW.lead_mean.max()),
    lead_iqr=[float(SW.lead_mean.quantile(.25)),float(SW.lead_mean.quantile(.75))],
    missed_max=float(SW.missed_rate.max()),
    best=SW.loc[SW.lead_mean.idxmax()].to_dict()),
    "kappa_note":"QCCI is exactly invariant to kappa: the capacity proxy is not an input to the index."}
s2=[]
for eta in [0.25,0.5,1.0]:
  for Dm in [1.0,1.5,3.0]:
    for d in test_d+test_h:
        Dh=np.minimum(d.pd+d.pp+d.pa,Dm)/Dm; Uh=np.maximum(0,1-eta*Dh)
        d["_E"]=np.clip(PAR["alpha"]*d.L+PAR["beta"]*d.S+PAR["gamma"]*Dh-PAR["delta"]*Uh,0,1)
    thr,rf,L,miss,nd=lead_at_far(test_d,test_h,"_E",EVENTS["skr_collapse"],0.05)
    s2.append(dict(eta=eta,Dmax=Dm,lead_mean=float(L.mean()),missed_rate=miss/max(1,nd)))
results["sensitivity"]["eta_Dmax"]=s2
print("weights: lead %.2f-%.2f, worst missed %.3f"%(SW.lead_mean.min(),SW.lead_mean.max(),SW.missed_rate.max()))
print("eta/Dmax: lead %.2f-%.2f"%(min(x["lead_mean"] for x in s2),max(x["lead_mean"] for x in s2)))

## 10. Critical-slowing-down precursors — Table 5, Fig. 6
Rolling variance and lag-1 autocorrelation, tested against a healthy-link null (Reviewer 1, Comment 5).

In [ ]:
r3=np.random.default_rng(SEED_AUX+7)
prec_d=[run("degrading",r3) for _ in range(200)]
prec_h=[run("healthy",  r3) for _ in range(200)]
W=15
def trend(x):
    t=np.arange(len(x)); x=np.asarray(x,float); ok=~np.isnan(x)
    return float(stats.kendalltau(t[ok],x[ok]).statistic) if ok.sum()>=8 else None
def stats_for(trajs, use_event):
    out={k:[] for k in ["var_S","ac1_S","var_F","ac1_F","var_QCCI","ac1_QCCI"]}
    for d in trajs:
        if use_event:
            tf=first_true(EVENTS["ent_death"](d))
            if tf is None or tf<45: continue
            sl=slice(0,tf)
        else: sl=slice(None)
        for nm,col in [("S","S"),("F","L"),("QCCI","QCCI")]:
            x=d[col].values[sl]; v=[];a=[]
            for i in range(W,len(x)):
                s=x[i-W:i]; v.append(float(s.var())); sm=s-s.mean()
                a.append(float(np.corrcoef(sm[:-1],sm[1:])[0,1]) if s.var()>1e-18 else np.nan)
            tv,ta=trend(v),trend(a)
            if tv is not None: out["var_"+nm].append(tv)
            if ta is not None: out["ac1_"+nm].append(ta)
    return out
P=stats_for(prec_d,True); N=stats_for(prec_h,False)
results["ews"]={}
for k in P:
    p=np.array(P[k]); n=np.array([x for x in N[k] if x==x])
    if len(p)<5 or len(n)<5: continue
    u=stats.mannwhitneyu(p,n,alternative="greater")
    results["ews"][k]=dict(n_deg=len(p),n_null=len(n),tau_deg=float(p.mean()),tau_null=float(n.mean()),
        frac_pos_deg=float((p>0).mean()),frac_pos_null=float((n>0).mean()),
        auc=float(u.statistic/(len(p)*len(n))),p_mwu=float(u.pvalue))
T5=pd.DataFrame(results["ews"]).T; T5.to_csv(OUT_DIR+"table5_precursors.csv")
print(T5[["tau_deg","tau_null","frac_pos_deg","auc","p_mwu"]].round(4).to_string())

## 11. Measurement-aware implementation — Table 6, Fig. 7
Three stabiliser settings, finite shots, Bell-diagonal reconstruction (Reviewer 1, Comment 7).

In [ ]:
results["measurement_aware"]={}
for shots in [100,500,2000,10000]:
    r2=np.random.default_rng(SEED_AUX+shots)
    th=[run("healthy",  r2,shots=shots) for _ in range(80)]
    td=[run("degrading",r2,shots=shots) for _ in range(80)]
    row={}
    for ind in ["QCCI_hat","ENT_hat","FID_L_hat","QCCI","FID_L"]:
        thr,rf,L,miss,nd=lead_at_far(td,th,ind,EVENTS["skr_collapse"],0.05)
        row[ind]=dict(realised_far=rf,lead_mean=float(L.mean()) if len(L) else None,
                      missed_rate=miss/max(1,nd))
    A=pd.concat(td+th)
    row["estimator_error"]=dict(
        S_rmse=float(np.sqrt(((A.S_hat-A.S)**2).mean())),
        L_rmse=float(np.sqrt(((A.L_hat-A.L)**2).mean())),
        QCCI_rmse=float(np.sqrt(((A.QCCI_hat-A.QCCI)**2).mean())),
        QCCI_corr=float(np.corrcoef(A.QCCI_hat,A.QCCI)[0,1]),
        SKR_rmse=float(np.sqrt(((A.SKR_hat-A.SKR)**2).mean())))
    results["measurement_aware"][str(shots)]=row
    print(shots, {k:round(v,4) for k,v in row["estimator_error"].items()})
T6=pd.DataFrame({k:v["estimator_error"] for k,v in results["measurement_aware"].items()}).T
T6.to_csv(OUT_DIR+"table6_measurement_aware.csv")

## 12. Extended link: loss, dark counts, entanglement swapping — Fig. 8

In [ ]:
def loss_dark_state(pd_,eta_loss,pdark):
    r=link_state(pd_,0,0)
    q=pdark*(1-eta_loss)/max(eta_loss+pdark*(1-eta_loss),1e-12)
    return (1-q)*r+q*np.eye(4)/4

def swap(rho1,rho2):
    """Bell measurement on the two inner qubits, post-selected on Phi+."""
    phi=np.zeros((2,2),dtype=complex); phi[0,0]=phi[1,1]=1/np.sqrt(2)
    K=np.zeros((4,16),dtype=complex)
    for a in range(2):
        for c in range(2):
            for b in range(2):
                K[a*2+c, ((a*2+b)*2+b)*2+c]+=phi.conj()[b,b]
    R=np.kron(rho1,rho2); out=K@R@K.conj().T
    return out/np.trace(out).real

rows=[]
for pd_ in np.linspace(0,0.6,13):
    r1=link_state(pd_,0,0); sw=swap(r1,link_state(pd_,0,0))
    rows.append(dict(pd=float(pd_),single_SKR=skr(r1),single_C=concurrence(r1),
                     swap_SKR=skr(sw),swap_C=concurrence(sw),swap_F=fidelity_pure_ref(sw)))
SWP=pd.DataFrame(rows); SWP.to_csv(OUT_DIR+"extended_swap.csv",index=False)
rows=[]
for pd_ in np.linspace(0,0.6,13):
    for eta in [1.0,0.5,0.1]:
        for pdk in [0.0,1e-3,1e-2]:
            r=loss_dark_state(pd_,eta,pdk)
            rows.append(dict(pd=float(pd_),eta=eta,pdark=pdk,SKR=skr(r),C=concurrence(r),F=fidelity_pure_ref(r)))
EXT=pd.DataFrame(rows); EXT.to_csv(OUT_DIR+"extended_loss.csv",index=False)
results["extended_link"]={"single_hop_skr_zero":cross(lambda p: skr(link_state(p,0,0))),
    "two_hop_skr_zero":cross(lambda p: skr(swap(link_state(p,0,0),link_state(p,0,0)))),
    "single_hop_ent_death":cross(lambda p: concurrence(link_state(p,0,0))-1e-12),
    "two_hop_ent_death":cross(lambda p: concurrence(swap(link_state(p,0,0),link_state(p,0,0)))-1e-12)}
print(results["extended_link"])

## 13. Manifest and completeness check

In [ ]:
results["environment"]=dict(python=platform.python_version(),numpy=np.__version__,
    pandas=pd.__version__,platform=platform.platform())
try:
    import qiskit; results["environment"]["qiskit"]=qiskit.__version__
except Exception: results["environment"]["qiskit"]="not installed"

with open(OUT_DIR+"manifest.json","w") as f: json.dump(results,f,indent=1,default=float)

EXPECTED=["table1_critical_points.csv","table2_lead_time.csv","table3_ablation.csv",
 "table4_weight_sensitivity.csv","table5_precursors.csv","table6_measurement_aware.csv",
 "grid_mixed_noise.csv","extended_swap.csv","extended_loss.csv","manifest.json"]
missing=[f for f in EXPECTED if not os.path.exists(OUT_DIR+f)]
print("MISSING:", missing if missing else "none")

print("\n=== HEADLINE NUMBERS AS REPORTED IN THE MANUSCRIPT ===")
m=results["matched_far"]["0.05"]["skr_collapse"]
for i in IND: print(f"  {i:11s} lead {m[i]['lead_mean']:.2f}  CI {[round(x,2) for x in m[i]['lead_ci']]}  missed {m[i]['missed_rate']:.3f}")
pv=results["paired"]["skr_collapse"]["FID_L"]
print(f"  QCCI vs 1-F: {pv['mean_diff']:+.3f} steps, p = {pv['p_ttest']:.3f}  <-- principal negative result")
print(f"  critical slowing down, AC1 fidelity AUC = {results['ews']['ac1_F']['auc']:.3f}")
print(f"  measurement-aware @500 shots, QCCI RMSE = {results['measurement_aware']['500']['estimator_error']['QCCI_rmse']:.4f}")